In [18]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os.path as path
import utils
import glob
import re

In [19]:
nuts_df = gpd.read_file(path.join(utils.raw_data_dir, "NUTS_RG_01M_2021_4326.shp"))

In [20]:
nuts_df = nuts_df.to_crs(epsg=3035)
nuts_df["area_km2"] = round(nuts_df.geometry.area / 1e6, 2)

In [21]:
population_data = pd.read_csv(path.join(utils.raw_data_dir, "estat_demo_r_pjangrp3.tsv"))
population_data = population_data[population_data["sex"] == "T"]
population_data = population_data[population_data["age"] == "TOTAL"]

In [22]:
last_col = population_data.columns[-1]

population_data[last_col] = population_data[last_col].apply(lambda x: [e for e in x.split("\t") if e != ": " ])
population_data["NUTS_ID"] = population_data[last_col].apply(lambda x: x[0])
population_data["population"] = population_data[last_col].apply(lambda x: x[-1])
population_data["population"] = population_data["population"].apply(lambda x: re.sub(r"[a-zA-Z]+", "", x).strip())
population_data["population"] = population_data["population"].apply(lambda x: int(x) if len(x) > 0 else np.nan)
population_data = population_data.drop(columns=[last_col, "sex", "age", "freq", "unit"])

In [23]:
nuts_df = pd.merge(nuts_df, population_data, on="NUTS_ID", how="outer")

In [24]:
nuts3_df = nuts_df[nuts_df["LEVL_CODE"] == 3].drop(columns=["LEVL_CODE", "MOUNT_TYPE", "URBN_TYPE", "COAST_TYPE"])

In [25]:
crop_profile_files = glob.glob(path.join(utils.intermediate_data_dir, "nuts3_crop_profile", "*.geojson"))
crop_profile_df = pd.concat([gpd.read_file(f) for f in crop_profile_files if not f.lower().endswith("uk.geojson")])

In [26]:
def calc_crop_area(in_dict):
    in_dict = eval(in_dict)
    out_dict = {}
    for k, v in in_dict.items():
        if k in [0, 65535]:
            # default value for non-cropland pixels is 0
            continue
        crop_name = utils.cropland_type_dict[k]
        # NOTE: this next step is important and deserves explanation
        # the raster size of the cropland dataset is _precisely_ 10x10m for all pixels, as defined by the CRS
        # thus, each pixel has an area of 100m^2. We get the total area in m^2 by multiplying the pixel value by 100
        # to get from m^2->km^2 we need to divide by 1.000*1.000, i.e. 1.000.000
        # in other words, we divide by 10.000 or 1e4
        area_km = round(v * 1e-4, 2)
        out_dict[crop_name] = area_km
    return out_dict

crop_profile_df["cropland_km2_by_type"] = crop_profile_df["crop_profile"].apply(calc_crop_area)
crop_profile_df["cropland_km2"] = crop_profile_df["cropland_km2_by_type"].apply(lambda x: round(sum(x.values()), 2))


In [27]:
crop_profile_df = crop_profile_df[["NUTS_ID", "cropland_km2", "cropland_km2_by_type"]]
nuts3_df = pd.merge(nuts3_df, crop_profile_df, on="NUTS_ID", how="outer")
nuts3_df["cropland_area_percent"] = round(100 * nuts3_df["cropland_km2"] / nuts3_df["area_km2"], 2)

In [28]:
nuts3_drought_data = utils.load_complex_geojson(
    path.join(utils.out_data_dir, "drought_days_nuts3.geojson")
)
nuts3_drought_data = nuts3_drought_data[
    [
        "NUTS_ID",
        "warning_days",
        "median_warning_days",
        "max_warning_days",
        "max_warning_days_year",
        "alert_days",
        "median_alert_days",
        "max_alert_days",
        "max_alert_days_year",
        "drought_days",
        "median_drought_days",
        "max_drought_days",
        "max_drought_days_year",
    ]
]

In [29]:
nuts3_df = gpd.pd.merge(nuts3_df, nuts3_drought_data, on="NUTS_ID", how="outer")
# nuts3_df = nuts3_df[
#     [
#         "NUTS_ID",
#         "CNTR_CODE",
#         "NUTS_NAME",
#         "population",
#         "area_km2",
#         "cropland_km2",
#         "cropland_area_percent",
#         "cropland_km2_by_type",
#         "NAME_LATN",
#         "median_drought_days",
#         "max_drought_days",
#         "max_drought_days_year",
#         "median_warning_days",
#         "max_warning_days",
#         "max_warning_days_year",
#         "median_alert_days",
#         "max_alert_days",
#         "max_alert_days_year",
#         "geometry",
#     ]
# ]
nuts3_df = nuts3_df.rename(columns = {colname:colname.lower() for colname in nuts3_df.columns})
nuts3_df.set_index("nuts_id", drop=True, inplace=True)
nuts3_regions_outside_cdi_raster = nuts3_df[nuts3_df["drought_days"].isna()]
print(f"dropping {len(nuts3_regions_outside_cdi_raster)} rows from countries {set(nuts3_regions_outside_cdi_raster['cntr_code'].values)} where no drought data was present")
nuts3_df = nuts3_df.dropna(subset="drought_days")

dropping 7 rows from countries {'PT', 'FR', 'NO'} where no drought data was present


In [30]:
country_df = gpd.read_file(utils.country_dir)
groups = nuts3_df.groupby("cntr_code")


In [31]:
country_df = country_df[country_df["CNTR_ID"].isin( list(groups.groups.keys()))]
country_df = country_df.set_index("CNTR_ID")

In [32]:
for cntr, group_df in groups:
    country_dict = {}
    
    
    sum_columns = ["area_km2", "cropland_km2", "population"]
    for colname in sum_columns:
        if group_df[colname].isna().sum() == 0:
            country_dict[colname] = group_df[colname].sum()
    
    for drought_metric in ["warning", "alert", "drought"]:
        # for each of the three drought-indicator metrics, we compute the median and max for the entire country based off the nuts3 stats:
        # read the metric days per year for each NUTS3 and store them in a 2d array of shape (n_nuts3, n_years), where n_years is always 14 (each year between 2012 and 2025).
        value_matrix = np.vstack(group_df[f"{drought_metric}_days"].values)
        # multiply the values for each NUTS3 region by the corresponding surface area in km2
        value_matrix = value_matrix * group_df["area_km2"].values.reshape(-1, 1)
        # sum over all nuts3 to get a 1d array with one value for each year.
        # Divide by the total surface area of the country to get areaa-weighted drought days
        value_matrix = value_matrix.sum(axis=0) / country_dict["area_km2"]

        # compute median, max and the year of the worst drought.
        country_dict[f"median_{drought_metric}_days"] = np.median(value_matrix).round(2)
        country_dict[f"max_{drought_metric}_days"] = np.max(value_matrix).round(2)
        country_dict[f"max_{drought_metric}_days_year"] = 2012 + np.argmax(value_matrix)

    for k, v in country_dict.items():
        country_df.loc[cntr, k] = v

In [33]:
country_df.columns

Index(['CNTR_NAME', 'NAME_ENGL', 'NAME_FREN', 'ISO3_CODE', 'SVRG_UN', 'CAPT',
       'EU_STAT', 'EFTA_STAT', 'CC_STAT', 'NAME_GERM', 'geometry', 'area_km2',
       'cropland_km2', 'population', 'median_warning_days', 'max_warning_days',
       'max_warning_days_year', 'median_alert_days', 'max_alert_days',
       'max_alert_days_year', 'median_drought_days', 'max_drought_days',
       'max_drought_days_year'],
      dtype='object')

In [34]:
country_df = country_df[['CNTR_NAME','area_km2',
       'cropland_km2', 'population', 'median_warning_days', 'max_warning_days',
       'max_warning_days_year', 'median_alert_days', 'max_alert_days',
       'max_alert_days_year', 'median_drought_days', 'max_drought_days',
       'max_drought_days_year']]
country_df = country_df.rename({"CNTR_NAME": "cntr_id"})

In [35]:
nuts3_df = nuts3_df.to_crs(epsg=4326)
nuts3_df.to_file(path.join(utils.out_data_dir, "nuts3_stats.geojson"))

In [36]:
for ctr, df in groups:
    df.drop(columns=["geometry", "cntr_code"]).to_excel(path.join(utils.out_data_dir, "nuts3_stats_by_country", f"{ctr}.xlsx"))

OSError: Cannot save file into a non-existent directory: '/Users/johannesgille/Desktop/CE_2026_4_drought/data_out/nuts3_stats_by_country'

In [ ]:
lau_lookup_table = {}


for ctr in list(set((nuts3_df["cntr_code"].values))):
    if ctr == "UK":
        pass
    else:
        lookup_df = pd.read_excel(path.join(utils.raw_data_dir, "EU-27-LAU-2023-NUTS-2021.xlsx"), sheet_name=ctr)
        break

In [ ]:
group_df

,cntr_code,nuts_name,population,area_km2,cropland_km2,cropland_area_percent,cropland_km2_by_type,name_latn,median_drought_days,max_drought_days,max_drought_days_year,median_warning_days,max_warning_days,max_warning_days_year,median_alert_days,max_alert_days,max_alert_days_year,geometry
nuts_id,,,,,,,,,,,,,,,,,,
AL011,AL,Dibër,104624,2470.31,23.12,0.94,"{'Maize': 10.12, 'Grapes': 0.13, 'Fruits': 8.5...",Dibër,113.21,248.16,2017.0,86.58,232.38,2020.0,11.69,58.02,2012.0,"POLYGON ((20.34672 41.87577, 20.34193 41.86729..."
AL012,AL,Durrës,222999,772.54,144.39,18.69,"{'Wheat': 19.74, 'Barley': 1.65, 'Maize': 28.9...",Durrës,153.29,269.56,2017.0,118.71,229.42,2017.0,11.76,62.81,2012.0,"POLYGON ((19.80665 41.56639, 19.81805 41.56442..."
AL013,AL,Kukës,60207,2391.45,8.19,0.34,"{'Other cereals': 0.4, 'Potatoes': 0.26, 'Rape...",Kukës,110.34,192.95,2017.0,99.16,186.05,2017.0,9.33,72.58,2012.0,"POLYGON ((20.14989 42.51392, 20.16495 42.50593..."
AL014,AL,Lezhë,96384,1662.35,111.27,6.69,"{'Wheat': 8.54, 'Barley': 1.55, 'Maize': 21.51...",Lezhë,127.13,266.82,2017.0,106.10,239.55,2017.0,13.18,49.57,2012.0,"POLYGON ((20.21222 42.02493, 20.21989 42.02303..."
AL015,AL,Shkodër,149496,3528.30,219.66,6.23,"{'Wheat': 5.85, 'Barley': 1.44, 'Maize': 35.03...",Shkodër,124.31,235.37,2017.0,98.37,226.12,2017.0,7.87,41.31,2012.0,"POLYGON ((19.75628 42.63384, 19.75005 42.62767..."
AL021,AL,Elbasan,226963,3314.10,125.91,3.80,"{'Wheat': 12.32, 'Barley': 7.88, 'Maize': 8.36...",Elbasan,69.72,228.31,2012.0,64.16,180.08,2012.0,5.77,48.23,2012.0,"POLYGON ((20.4494 41.38775, 20.45035 41.37053,..."
AL022,AL,Tiranë,759981,1652.22,135.24,8.19,"{'Wheat': 21.91, 'Barley': 3.73, 'Maize': 18.1...",Tiranë,148.26,261.30,2017.0,127.26,244.93,2017.0,12.13,42.41,2012.0,"POLYGON ((19.99507 41.44384, 19.99799 41.43036..."
AL031,AL,Berat,138926,1801.97,78.18,4.34,"{'Wheat': 8.7, 'Barley': 4.88, 'Maize': 4.81, ...",Berat,90.76,264.06,2025.0,78.62,249.90,2025.0,11.06,26.73,2012.0,"POLYGON ((19.96762 40.87032, 19.96937 40.86404..."
AL032,AL,Fier,233215,1881.63,475.27,25.26,"{'Wheat': 94.73, 'Barley': 38.71, 'Maize': 66....",Fier,199.85,334.84,2017.0,179.04,322.82,2017.0,4.97,55.96,2012.0,"POLYGON ((19.6006 41.06751, 19.61202 41.06339,..."
